# Digital DNA — Music Metadata API Exploration

This notebook explores the MusicBrainz API as an external metadata source for the Digital DNA personal music analytics platform.

## Goals
- Connect to a REST API using Python
- Retrieve music metadata in JSON format
- Inspect the API response structure
- Identify fields useful for the Digital DNA pipeline

In [2]:
import requests
import pandas as pd

In [3]:
base_url = "https://musicbrainz.org/ws/2/recording/"

In [4]:
params = {
    "query": 'recording:"Espresso" AND artist:"Sabrina Carpenter"',
    "fmt": "json",
    "limit": 5
}

In [5]:
headers = {
    "User-Agent": "DigitalDNA/1.0 (personal data analytics project)"
}

response = requests.get(
    base_url,
    params=params,
    headers=headers,
    timeout=10
)

In [6]:
print(response.status_code)

200


In [7]:
print(response.text[:500])

{"created":"2026-09-07T01:39:06.097Z","count":32,"offset":0,"recordings":[{"id":"b013776b-4703-4f41-8743-25ac90abc623","score":100,"artist-credit-id":"b49d5a60-c845-344f-a0c5-ef5c1c9dd66f","title":"Espresso","length":175000,"disambiguation":"Dolby Atmos mix, explicit","video":null,"artist-credit":[{"name":"Sabrina Carpenter","artist":{"id":"1882fe91-cdd9-49c9-9956-8e06a3810bd4","name":"Sabrina Carpenter","sort-name":"Carpenter, Sabrina","aliases":[{"sort-name":"Carpenter, Sabrina","type-id":"894


## API Error Handling

The initial MusicBrainz API request returned HTTP 503, indicating that the service was temporarily unavailable. This is a server-side availability issue rather than a client-side request error.

A production pipeline should account for temporary API failures with retry logic, backoff, and response validation.

In [9]:
import time

def get_with_retry(url, params, headers, retries=3, wait_seconds=2):
    for attempt in range(retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=10
        )

        if response.status_code == 200:
            return response

        print(
            f"Attempt {attempt + 1} failed "
            f"with status code {response.status_code}"
        )

        time.sleep(wait_seconds)

    return response

In [10]:
response = get_with_retry(
    base_url,
    params,
    headers
)

print("Final status code:", response.status_code)

ReadTimeout: HTTPSConnectionPool(host='musicbrainz.org', port=443): Read timed out. (read timeout=10)

In [ ]:
data = response.json()

print(type(data))
print(data.keys())

In [ ]:
recordings = data["recordings"]

print(type(recordings))
print("Number of results:", len(recordings))

In [ ]:
recordings[0]

In [ ]:
for i, recording in enumerate(recordings):
    print("Result:", i)
    print("Title:", recording.get("title"))
    print("Score:", recording.get("score"))
    print("First release:", recording.get("first-release-date"))
    print("Disambiguation:", recording.get("disambiguation"))
    print("-" * 50)

In [ ]:
results_df = pd.DataFrame([
    {
        "title": recording.get("title"),
        "score": recording.get("score"),
        "first_release_date": recording.get("first-release-date"),
        "disambiguation": recording.get("disambiguation")
    }
    for recording in recordings
])

results_df

In [ ]:
results_df = pd.DataFrame([
    {
        "recording_id": recording.get("id"),
        "title": recording.get("title"),
        "artist": recording.get("artist-credit", [{}])[0]
                           .get("artist", {})
                           .get("name"),
        "score": recording.get("score"),
        "first_release_date": recording.get("first-release-date"),
        "disambiguation": recording.get("disambiguation")
    }
    for recording in recordings
])

results_df

In [ ]:
results_df["is_potential_match"] = (
    (results_df["title"].str.lower() == "espresso") &
    (results_df["artist"].str.lower() == "sabrina carpenter")
)

results_df

In [ ]:
excluded_terms = [
    "live",
    "dj-mix",
    "lyric video"
]

results_df["is_preferred_version"] = ~results_df["disambiguation"].fillna("").str.lower().apply(
    lambda text: any(term in text for term in excluded_terms)
)

results_df

In [ ]:
results_df["clean_disambiguation"] = (
    results_df["disambiguation"]
    .fillna("")
    .str.lower()
    .str.replace("‐", "-", regex=False)
)

results_df[["disambiguation", "clean_disambiguation"]]

In [ ]:
results_df["is_preferred_version"] = ~results_df["clean_disambiguation"].apply(
    lambda text: any(term in text for term in excluded_terms)
)

results_df[
    ["title", "artist", "disambiguation", "is_preferred_version"]
]

In [ ]:
def get_rejection_reason(text):
    for term in excluded_terms:
        if term in text:
            return f"Excluded variant: {term}"
    return None

results_df["rejection_reason"] = results_df["clean_disambiguation"].apply(
    get_rejection_reason
)

results_df[
    [
        "title",
        "artist",
        "disambiguation",
        "is_preferred_version",
        "rejection_reason"
    ]
]

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
from src.ingestion.musicbrainz_api import search_recording

In [3]:
test_data = search_recording(
    "Espresso",
    "Sabrina Carpenter"
)

print(type(test_data))
print(test_data.keys())

<class 'dict'>
dict_keys(['created', 'count', 'offset', 'recordings'])


In [4]:
from src.transformation.recording_transform import recordings_to_dataframe

In [5]:
test_df = recordings_to_dataframe(test_data)
test_df

,recording_id,title,artist,score,first_release_date,disambiguation
0,4c567421-9895-4ee0-a442-9c63cab23e07,Espresso,Sabrina Carpenter,100,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington..."
1,cea42ffa-6d93-406c-964c-c4eb47c0a18b,Espresso,Sabrina Carpenter,100,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix
2,c733559e-0294-40ca-b6a8-b4b022fdef9e,Espresso,Sabrina Carpenter,100,2024-04-13,"live, 2024‐04‐12: Coachella Stage, Indio, CA, USA"
3,80bb466a-f303-486e-adf0-b0d9968b5c32,Espresso,Sabrina Carpenter,100,2024-05-03,"Dolby Atmos mix, clean"
4,554d72ad-6a60-478f-9755-b2b969d2dced,Espresso,Sabrina Carpenter,100,2024-08-26,track by track commentary


In [6]:
from src.validation.recording_validation import (
    normalize_disambiguation,
    get_rejection_reason,
    is_preferred_version,
    get_match_priority
)

In [7]:
test_df["is_preferred_version"] = test_df["disambiguation"].apply(
    is_preferred_version
)

test_df["rejection_reason"] = test_df["disambiguation"].apply(
    get_rejection_reason
)

test_df[
    [
        "title",
        "artist",
        "disambiguation",
        "is_preferred_version",
        "rejection_reason"
    ]
]

,title,artist,disambiguation,is_preferred_version,rejection_reason
0,Espresso,Sabrina Carpenter,"live, 2024-12-20: NPR Music Office, Washington...",False,Excluded variant: live
1,Espresso,Sabrina Carpenter,part of “Today’s Hits: July 2024” DJ‐mix,False,Excluded variant: dj-mix
2,Espresso,Sabrina Carpenter,"live, 2024‐04‐12: Coachella Stage, Indio, CA, USA",False,Excluded variant: live
3,Espresso,Sabrina Carpenter,"Dolby Atmos mix, clean",True,None
4,Espresso,Sabrina Carpenter,track by track commentary,False,Excluded variant: commentary


In [8]:
test_df[
    [
        "recording_id",
        "title",
        "artist",
        "first_release_date",
        "disambiguation"
    ]
]

,recording_id,title,artist,first_release_date,disambiguation
0,4c567421-9895-4ee0-a442-9c63cab23e07,Espresso,Sabrina Carpenter,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington..."
1,cea42ffa-6d93-406c-964c-c4eb47c0a18b,Espresso,Sabrina Carpenter,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix
2,c733559e-0294-40ca-b6a8-b4b022fdef9e,Espresso,Sabrina Carpenter,2024-04-13,"live, 2024‐04‐12: Coachella Stage, Indio, CA, USA"
3,80bb466a-f303-486e-adf0-b0d9968b5c32,Espresso,Sabrina Carpenter,2024-05-03,"Dolby Atmos mix, clean"
4,554d72ad-6a60-478f-9755-b2b969d2dced,Espresso,Sabrina Carpenter,2024-08-26,track by track commentary


In [18]:
test_df["match_priority"] = test_df["disambiguation"].apply(
    get_match_priority
)

ranked_df = test_df.sort_values(
    by="match_priority",
    ascending=False
)

ranked_df[
    [
        "title",
        "artist",
        "first_release_date",
        "disambiguation",
        "is_preferred_version",
        "rejection_reason",
        "match_priority"
    ]
]

,title,artist,first_release_date,disambiguation,is_preferred_version,rejection_reason,match_priority
3,Espresso,Sabrina Carpenter,2024-05-03,"Dolby Atmos mix, clean",True,None,1
0,Espresso,Sabrina Carpenter,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington...",False,Excluded variant: live,0
1,Espresso,Sabrina Carpenter,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix,False,Excluded variant: dj-mix,0
2,Espresso,Sabrina Carpenter,2024-04-13,"live, 2024‐04‐12: Coachella Stage, Indio, CA, USA",False,Excluded variant: live,0
4,Espresso,Sabrina Carpenter,2024-08-26,track by track commentary,False,Excluded variant: commentary,0


In [20]:
import importlib
import src.ingestion.musicbrainz_api as musicbrainz_api

importlib.reload(musicbrainz_api)

search_recording = musicbrainz_api.search_recording

In [24]:
test_data = search_recording(
    "Espresso",
    "Sabrina Carpenter"
)

test_df = recordings_to_dataframe(test_data)

print("Number of candidates:", len(test_df))

Attempt 1 failed with status code 503
Number of candidates: 20


In [26]:
import json

sample_path = project_root / "data" / "sample" / "musicbrainz_espresso_sample.json"

with open(sample_path, "w") as file: 
    json.dump(test_data, file, indent=2)

print(sample_path)

/Users/ashlyngrace/Documents/Projects/digital-dna/data/sample/musicbrainz_espresso_sample.json


In [28]:
with open(sample_path, "r") as file:
    saved_sample_data = json.load(file)

saved_sample_df = recordings_to_dataframe(saved_sample_data)

saved_sample_df.head()

,recording_id,title,artist,score,first_release_date,disambiguation
0,4c567421-9895-4ee0-a442-9c63cab23e07,Espresso,Sabrina Carpenter,100,2025-02-21,"live, 2024-12-20: NPR Music Office, Washington..."
1,cea42ffa-6d93-406c-964c-c4eb47c0a18b,Espresso,Sabrina Carpenter,100,2024-07-05,part of “Today’s Hits: July 2024” DJ‐mix
2,9867e32e-89ff-4b0c-a101-7d9126e114cd,Espresso,Sabrina Carpenter,100,2024-05-03,clean
3,a59823d1-3570-40eb-9317-227ded561779,Espresso,Sabrina Carpenter,100,None,"live, 2024-05-18: Saturday Night Live"
4,b013776b-4703-4f41-8743-25ac90abc623,Espresso,Sabrina Carpenter,100,2024-04-11,"Dolby Atmos mix, explicit"


In [30]:
saved_sample_df["is_preferred_version"] = saved_sample_df["disambiguation"].apply(
    is_preferred_version
)

saved_sample_df["rejection_reason"] = saved_sample_df["disambiguation"].apply(
    get_rejection_reason
)

saved_sample_df["match_priority"] = saved_sample_df["disambiguation"].apply(
    get_match_priority
)

saved_sample_ranked = saved_sample_df.sort_values(
    by="match_priority",
    ascending=False
)

saved_sample_ranked[
    [
        "title",
        "artist",
        "first_release_date",
        "disambiguation",
        "is_preferred_version",
        "rejection_reason",
        "match_priority"
    ]
]

,title,artist,first_release_date,disambiguation,is_preferred_version,rejection_reason,match_priority
19,Espresso (Ced rework),Sabrina Carpenter,2024-04-20,None,True,None,2
13,Espresso (espressooooo version),Sabrina Carpenter,2024-05-17,None,True,None,2
16,Espresso (decaf version),Sabrina Carpenter,2024-05-17,None,True,None,2
14,Espresso (on vacation),Sabrina Carpenter,2024-05-17,None,True,None,2
11,Espresso,Sabrina Carpenter,2024-04-12,None,True,None,2
12,Espresso (Espressooooo Version),Sabrina Carpenter,2025-11-28,None,True,None,2
2,Espresso,Sabrina Carpenter,2024-05-03,clean,True,None,1
18,Espresso (decaf version),Sabrina Carpenter,2024-05-17,Dolby Atmos mix,True,None,1
4,Espresso,Sabrina Carpenter,2024-04-11,"Dolby Atmos mix, explicit",True,None,1
17,Espresso (on vacation),Sabrina Carpenter,2024-05-17,Dolby Atmos mix,True,None,1


In [32]:
import importlib
import src.validation.recording_validation as recording_validation

importlib.reload(recording_validation)

get_rejection_reason = recording_validation.get_rejection_reason
is_preferred_version = recording_validation.is_preferred_version
get_match_priority = recording_validation.get_match_priority

In [34]:
saved_sample_df["match_priority"] = saved_sample_df.apply(
    lambda row: get_match_priority(
        row["title"],
        "Espresso",
        row["disambiguation"]
    ),
    axis=1
)

saved_sample_ranked = saved_sample_df.sort_values(
    by="match_priority",
    ascending=False
)

saved_sample_ranked[
    [
        "title",
        "artist",
        "first_release_date",
        "disambiguation",
        "match_priority"
    ]
]

,title,artist,first_release_date,disambiguation,match_priority
11,Espresso,Sabrina Carpenter,2024-04-12,None,3
2,Espresso,Sabrina Carpenter,2024-05-03,clean,2
4,Espresso,Sabrina Carpenter,2024-04-11,"Dolby Atmos mix, explicit",2
7,Espresso,Sabrina Carpenter,2024-04-11,explicit,2
9,Espresso,Sabrina Carpenter,2024-05-03,"Dolby Atmos mix, clean",2
19,Espresso (Ced rework),Sabrina Carpenter,2024-04-20,None,1
13,Espresso (espressooooo version),Sabrina Carpenter,2024-05-17,None,1
18,Espresso (decaf version),Sabrina Carpenter,2024-05-17,Dolby Atmos mix,1
17,Espresso (on vacation),Sabrina Carpenter,2024-05-17,Dolby Atmos mix,1
16,Espresso (decaf version),Sabrina Carpenter,2024-05-17,None,1


In [36]:
import importlib
import src.validation.recording_validation as recording_validation

importlib.reload(recording_validation)

select_best_match = recording_validation.select_best_match

In [38]:
best_match = select_best_match(
    saved_sample_df,
    "Espresso"
)

best_match

recording_id            48ee49e2-b4db-47fd-96de-5ec2ac542a48
title                                               Espresso
artist                                     Sabrina Carpenter
score                                                     99
first_release_date                                2024-04-12
disambiguation                                          None
is_preferred_version                                    True
rejection_reason                                        None
match_priority                                             3
Name: 11, dtype: object

In [42]:
import importlib
import src.transformation.recording_transform as recording_transform

importlib.reload(recording_transform)

process_recording_matches = recording_transform.process_recording_matches

In [44]:
best_match_from_sample = process_recording_matches(
    saved_sample_data,
    "Espresso"
)

best_match_from_sample

recording_id          48ee49e2-b4db-47fd-96de-5ec2ac542a48
title                                             Espresso
artist                                   Sabrina Carpenter
score                                                   99
first_release_date                              2024-04-12
disambiguation                                        None
match_priority                                           3
Name: 11, dtype: object